# Parte 2a: Modelo A (CNN custom)

> **Notebook auxiliar.** Se usa para trabajar en paralelo. Su contenido se integra en `2-train.ipynb`, que es la que se entrega.
> La lógica va en `src/lib/` (`lib.models`, `lib.training`); acá solo se llaman funciones. Al terminar, guarda los embeddings con `save_embeddings` (contrato en `src/lib/evaluation/embedding_store.py`).

**Colab:** se entrena en Google Colab (GPU T4). La primera celda clona el repo, baja las caras alineadas de Drive y deja los checkpoints en Drive, así una sesión cortada se retoma desde la última época. En local la celda solo usa el repo actual.

In [ ]:
# 0. Entorno: Colab (repo clonado + datos de Drive) o local
import sys
from pathlib import Path

EXPERIMENT = "exp4_arcface_finetune"  # cada corrida guarda sus checkpoints en su propia carpeta
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/valentinlujambio/tuia-face-recognition-app.git"
    BRANCH = "feature/tp-reconocimiento-facial"
    ROOT = Path("/content/tuia-face-recognition-app")
    DRIVE_DIR = Path("/content/drive/MyDrive/tp1")
    CKPT_DIR = DRIVE_DIR / "checkpoints" / EXPERIMENT

    if ROOT.exists():
        !git -C {ROOT} pull --ff-only
    else:
        !git clone --depth 1 --branch {BRANCH} {REPO_URL} {ROOT}
    %cd {ROOT}
    !pip install -q albumentations==2.0.8 onnx onnxruntime

    if not (ROOT / "data/processed/aligned").exists():
        archive = next(DRIVE_DIR.glob("aligned.*"))
        !mkdir -p data/processed
        if archive.suffix == ".rar":
            !apt-get -qq install -y unrar > /dev/null
            !unrar x -o+ -inul "{archive}" data/processed/
        else:
            !unzip -q -o "{archive}" -d data/processed/
else:
    ROOT = Path.cwd()
    CKPT_DIR = ROOT / "results" / "custom_cnn" / "checkpoints" / EXPERIMENT

sys.path.insert(0, str(ROOT / "src"))
print("ROOT:", ROOT, "| checkpoints:", CKPT_DIR)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from lib.dataset.augmentation import build_train_augmentation
from lib.evaluation import EmbeddingSet, results_path, save_embeddings
from lib.models import ArcFaceHead, CustomCNNConfig, CustomFaceCNN, SoftmaxHead, count_parameters
from lib.training import CheckpointStore, EarlyStopping, FaceDataset, LabelEncoder, Trainer, load_faces, set_seed
from lib.vision import InputNormalization

CONFIG = json.loads((ROOT / "splits" / "config.json").read_text(encoding="utf-8"))
ALIGNED_ROOT = ROOT / CONFIG["aligned_root"]
SEED = CONFIG["seed"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)

splits = pd.read_csv(ROOT / "splits" / "splits.csv")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
splits["split"].value_counts()

## 1. Datos
- `train`: entrenamiento (80 identidades). `val_closed`: mismas identidades, para las curvas de loss/accuracy.
- `val_open`: identidades **no vistas**; su AUC de verificación elige el mejor checkpoint y dispara el early stopping, porque la app reconoce personas registradas después del entrenamiento.
- Augmentation de la Parte 1 solo en train; mismo preprocesamiento que la app (`InputNormalization`: RGB y `(x - 127.5) / 127.5`).
- Las ~13k caras de 112×112 entran en RAM (~0,5 GB en uint8), así el DataLoader no lee disco.

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2

rows = {name: splits[splits["split"] == name].reset_index(drop=True) for name in ("train", "val_closed", "val_open")}
encoder = LabelEncoder(rows["train"]["identity"])
open_encoder = LabelEncoder(rows["val_open"]["identity"])  # solo para saber qué pares son la misma persona
faces = {name: load_faces(ALIGNED_ROOT, df["aligned_path"]) for name, df in rows.items()}

normalization = InputNormalization()
datasets = {
    "train": FaceDataset(faces["train"], encoder.encode(rows["train"]["identity"]), normalization,
                         augment=build_train_augmentation()),
    "val_closed": FaceDataset(faces["val_closed"], encoder.encode(rows["val_closed"]["identity"]), normalization),
    "val_open": FaceDataset(faces["val_open"], open_encoder.encode(rows["val_open"]["identity"]), normalization),
}
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                     persistent_workers=NUM_WORKERS > 0)
loaders = {
    "train": DataLoader(datasets["train"], shuffle=True, drop_last=True, **loader_kwargs),
    "val_closed": DataLoader(datasets["val_closed"], shuffle=False, **loader_kwargs),
    "val_open": DataLoader(datasets["val_open"], shuffle=False, **loader_kwargs),
}
pd.DataFrame({name: {"caras": len(ds), "identidades": len(np.unique(ds.labels))} for name, ds in datasets.items()}).T

## 2. Arquitectura
ResNet chica (`src/lib/models/custom_cnn.py`, importable desde el backend):

| Bloque | Salida | Detalle |
|---|---|---|
| Stem | 32×112×112 | conv 3×3 + BN + PReLU |
| Etapa 1–4 | 32×56², 64×28², 128×14², 256×7² | 2 bloques residuales por etapa, el primero con stride 2 |
| Embedding | 128 | BN → Dropout → global average pooling → FC → BN |

- **Profundidad:** 4 etapas (sugerencia del enunciado, 4–6 bloques) llevan 112 → 7; los atajos residuales permiten 8 bloques sin problemas de gradiente.
- **BatchNorm** tras cada conv: estabiliza el entrenamiento desde cero y tolera lr más altos.
- **Dropout** antes del FC: regulariza con solo ~110 imágenes por identidad.
- **Global pooling:** 256 parámetros de entrada al FC en vez de 7·7·256 = 12.544 (FC de 33k vs 1,6M parámetros); con pocos datos reduce el sobreajuste.
- **Embedding de 128-d** sin normalizar; la normalización L2 la hace quien lo consume (igual que ArcFace en la app).
- **Cabeza** (`src/lib/models/heads.py`, solo para entrenar, no se exporta):
  - `SoftmaxHead` (exp 1 y 2): FC 128 → 80 identidades + cross-entropy.
  - `ArcFaceHead` (exp 3 y 4, como en la clase 4): normaliza embedding y pesos, logit = s·cos(θ + m) para la clase correcta (s = 30, m = 0,5). El margen angular obliga a que cada identidad quede compacta y separada de las demás, que es lo que mide la verificación por coseno.
- **Exp 4 (fine-tuning):** ArcFace desde cero no convergió (exp 3), así que se parte del extractor del exp 1 (`INIT_FROM`) y la cabeza ArcFace se inicializa con el centroide normalizado de cada identidad de train. Así la clase correcta ya "gana" desde la primera época y el margen solo compacta lo que softmax ya separó.

In [ ]:
model_config = CustomCNNConfig(channels=(32, 64, 128, 256), blocks_per_stage=2, embedding_dim=128, dropout=0.2)
model = CustomFaceCNN(model_config)
HEAD = "arcface"  # "softmax" (exp 1 y 2) o "arcface" (exp 3 y 4)
INIT_FROM = "exp1_softmax_sgd"  # experimento cuyo best.pt inicializa el extractor; None = desde cero
if INIT_FROM:
    model = CustomFaceCNN.from_checkpoint(CKPT_DIR.parent / INIT_FROM / "best.pt")
head = (ArcFaceHead(model_config.embedding_dim, num_classes=len(encoder), scale=30.0, margin=0.5)
        if HEAD == "arcface" else SoftmaxHead(model_config.embedding_dim, num_classes=len(encoder)))

print(model)
print(f"Parámetros extractor: {count_parameters(model):,} | cabeza: {count_parameters(head):,}")

## 3. Hiperparámetros

| | Valor | Por qué |
|---|---|---|
| Pérdida | cross-entropy sobre la cabeza (softmax en exp 1–2, ArcFace en exp 3) | enfoque 1 y 2 del enunciado |
| Optimizador | SGD, momentum 0,9, Nesterov | en exp 1 vs 2 rindió mejor que AdamW (AUC 0,893 vs 0,882) |
| Learning rate (exp 4) | 0,01 con warmup 1 época + coseno, 25 épocas, paciencia 10 | fine-tuning: lr 10x menor para no destruir lo aprendido en el exp 1 |
| Learning rate | 0,1 con warmup lineal 2 épocas + coseno | warmup evita el arranque inestable con lr alto |
| Weight decay | 5e-4 | regularización L2 |
| Batch size | 128 | entra holgado en una T4 con AMP |
| Épocas | 60 máx., early stopping con paciencia 15 | sobre AUC de verificación en val_open; en exp 2 paciencia 10 cortó antes de la fase de lr bajo del coseno |
| AMP | float16 en GPU | ~2x más rápido en T4 |

In [ ]:
EPOCHS = 25
LR = 0.01
WARMUP_EPOCHS = 1
WEIGHT_DECAY = 5e-4
PATIENCE = 10

params = list(model.parameters()) + list(head.parameters())
optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS),
    ],
    milestones=[WARMUP_EPOCHS],
)
trainer = Trainer(model, head, optimizer, scheduler, DEVICE, amp=True)
store = CheckpointStore(CKPT_DIR)
early_stopping = EarlyStopping(patience=PATIENCE)

if INIT_FROM and HEAD == "arcface":
    # Cabeza alineada con las identidades: centroide de los embeddings de train (sin augmentation, sin shuffle).
    train_plain = DataLoader(FaceDataset(faces["train"], datasets["train"].labels, normalization), shuffle=False, **loader_kwargs)
    head.init_from_centroids(torch.from_numpy(trainer.embed(train_plain)), torch.from_numpy(datasets["train"].labels))

## 4. Entrenamiento
Cada época guarda `last.pt` (estado completo) e `history.json` en Drive; `best.pt` cuando mejora el AUC de val_open (marcado con `*`).
Si Colab corta la sesión, volver a correr la notebook reanuda desde la última época. Para empezar de cero: `resume=False` o borrar la carpeta de checkpoints.

In [ ]:
history = trainer.fit(
    loaders["train"], loaders["val_closed"], loaders["val_open"],
    epochs=EPOCHS, early_stopping=early_stopping, store=store, resume=True,
)

## 5. Curvas de entrenamiento
Loss y accuracy de train vs. validation (val_closed) y AUC/EER de verificación en val_open.

In [ ]:
hist = pd.DataFrame(json.loads(store.history_path.read_text(encoding="utf-8")))
best_epoch = int(hist.loc[hist["val_open_auc"].idxmax(), "epoch"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (metric, title) in zip(axes[:2], [("loss", "Loss"), ("acc", "Accuracy")]):
    ax.plot(hist["epoch"], hist[f"train_{metric}"], label="train")
    ax.plot(hist["epoch"], hist[f"val_{metric}"], label="val_closed")
    ax.set_title(title)
axes[2].plot(hist["epoch"], hist["val_open_auc"], label="AUC")
axes[2].plot(hist["epoch"], hist["val_open_eer"], label="EER")
axes[2].set_title("Verificación en val_open")
for ax in axes:
    ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("época")
    ax.legend()
plt.tight_layout()
plt.show()
hist.loc[hist["epoch"] == best_epoch]

## 6. Experimentos

Mejor época de cada corrida (checkpoints en `MyDrive/tp1/checkpoints/<experimento>`). Con ArcFace la val loss se calcula sobre s·cos θ sin margen, así que no es comparable en escala con la de softmax; la comparación entre experimentos se hace con AUC/EER de val_open.

| # | Experimento | Cambio | Mejor época | Val loss | Val acc | AUC val_open | EER val_open | Conclusión |
|---|---|---|---|---|---|---|---|---|
| 1 | `exp1_softmax_sgd` | Baseline: softmax + SGD (lr 0,1, momentum 0,9, wd 5e-4), 4 etapas × 2 bloques, emb 128, dropout 0,2 | 41 / 51 | 0,369 | 0,909 | 0,893 | 0,188 | Train acc 0,99 vs val 0,91: sobreajuste leve. AUC val_open estancado en ~0,88–0,89 desde la época 30 mientras val_closed sigue mejorando: softmax separa las 80 identidades de train pero no generaliza tanto a personas nuevas. |
| 2 | `exp2_softmax_adamw` | Igual que 1 con AdamW (lr 1e-3, wd 1e-4), como en la cátedra | 27 / 38 | 0,506 | 0,864 | 0,882 | 0,200 | Algo peor que SGD. El early stopping (paciencia 10) cortó con lr = 3,4e-4 y val acc todavía subiendo: no llegó a la fase de lr bajo, donde SGD logró su mejor época. Igual que en 1, el AUC val_open se estanca en ~0,88: el límite es la pérdida softmax, no el optimizador. |
| 3 | `exp3_arcface_sgd` | Como 1 pero con cabeza ArcFace (s = 30, m = 0,5) y paciencia 15 | 0 / 15 | 4,27 | 0,037 | 0,566 | 0,451 | **No convergió.** Loss trabada en ~11,4 desde la época 2 y AUC ≈ 0,5 (azar). Desde cero todos los ángulos están cerca de 90° y con m = 0,5 la clase correcta nunca supera a las demás: la red colapsa y deja de aprender. Falla conocida de ArcFace con pocos datos y sin preentrenamiento. |
| 4 | `exp4_arcface_finetune` | ArcFace partiendo del extractor del exp 1, cabeza inicializada con centroides, lr 0,01, 25 épocas | 0 / 10 | 0,407 | 0,919 | **0,903** | **0,178** | **Mejor modelo.** En la época 0 (warmup, lr 0,001) ArcFace compacta los embeddings y supera al exp 1 en las tres métricas. Al subir a lr 0,01 el AUC cae a ~0,84–0,86 y no se recupera: lr demasiado alto para fine-tuning, desarma lo aprendido. Siguiente prueba natural: fine-tuning con lr 0,001 constante. |

**Elegido:** `exp4_arcface_finetune` (mejor AUC y EER en val_open).

## 7. Exportación
Se exporta el `best.pt` del experimento elegido con `export_onnx` (`src/lib/models/export.py`): batch dinámico, salida `(N, 128)` sin normalizar, y verificación de que onnxruntime da lo mismo que torch.
`models/custom_cnn.onnx` no se versiona (`*.onnx` en `.gitignore`): se copia a Drive para compartirlo con link público y ponerlo en el README.
En la app: `MODEL_NAME=custom_cnn.onnx`, `EMBEDDING_DIM=128`, mismo preprocesamiento (`InputNormalization`) y normalización L2.

In [ ]:
import shutil
import time

import onnxruntime

from lib.models import export_onnx

BEST_EXPERIMENT = "exp4_arcface_finetune"
best_model = CustomFaceCNN.from_checkpoint(CKPT_DIR.parent / BEST_EXPERIMENT / "best.pt")
onnx_path = export_onnx(best_model, ROOT / "models" / "custom_cnn.onnx")
if IN_COLAB:
    shutil.copy(onnx_path, DRIVE_DIR / onnx_path.name)

# Costo computacional (para la comparación con el Modelo B)
session = onnxruntime.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
one_face = normalization([faces["val_open"][0]])
session.run(None, {"input": one_face})  # warmup
t0 = time.perf_counter()
for _ in range(100):
    session.run(None, {"input": one_face})
pd.Series({
    "parámetros": f"{count_parameters(best_model):,}",
    "tamaño .onnx (MB)": round(onnx_path.stat().st_size / 1e6, 2),
    "ms por cara (CPU, batch 1)": round((time.perf_counter() - t0) * 10, 2),
    "dimensión del embedding": best_model.embedding_dim,
})

## 8. Embeddings para la evaluación
Se calculan sobre todos los splits salvo `train` y se guardan con el contrato común.

In [ ]:
from lib.vision import OnnxFaceEmbedder

# Mismo camino que la app: .onnx + InputNormalization + L2.
embedder = OnnxFaceEmbedder(onnx_path, normalization=normalization)
eval_rows = splits[splits["split"] != "train"].reset_index(drop=True)
embeddings = embedder.embed(load_faces(ALIGNED_ROOT, eval_rows["aligned_path"]))

npz_path = save_embeddings(
    EmbeddingSet(
        model="custom_cnn",
        paths=eval_rows["aligned_path"].to_numpy(),
        labels=eval_rows["identity"].to_numpy(),
        splits=eval_rows["split"].to_numpy(),
        embeddings=embeddings,
    ),
    results_path(ROOT, "custom_cnn"),
)
if IN_COLAB:
    shutil.copy(npz_path, DRIVE_DIR / "embeddings_custom_cnn.npz")
print(npz_path, embeddings.shape)
eval_rows["split"].value_counts()